# Advanced 02 - Header Manipulation & Pattern Matching

In this notebook:
- Learn to add, remove, and modify SIP headers
- Practice regex pattern matching
- Use `subst` functions to transform messages
- Learn practical debugging techniques

---

## What is SIP Header Manipulation?

A SIP Proxy modifies headers before forwarding messages:
- **Add**: Custom headers like `X-Custom`, `P-Charge-Info`
- **Remove**: Unnecessary or security-sensitive headers
- **Modify**: Pattern-based text replacement with `subst()`

This is one of the most common tasks in production.

## 1. append_hf — Adding Headers

`append_hf("Header: value")` appends a SIP header to the end of the message.

Practical use: adding billing info, internal routing info, etc.

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=h1
To: <sip:1002@example.com>

In [ ]:
# Add custom headers
append_hf("X-Source-IP: $si");
append_hf("X-Custom-Route: internal");
xlog("Added custom headers");

# Pass caller info in a header
$var(caller) = $(fu{uri.user});
append_hf("P-Charge-Info: sip:$var(caller)@example.com");
xlog("Added P-Charge-Info for billing: $var(caller)");

## 2. remove_hf — Removing Headers

`remove_hf("Header")` removes a specific header from the message.

Practical use: removing sensitive headers before sending to external networks.

In [ ]:
# Remove unnecessary headers
remove_hf("X-Internal-Info");
remove_hf("X-Debug");
xlog("Removed internal headers before relaying");

## 3. is_present_hf — Checking Header Presence

`is_present_hf("Header")` checks if a specific header exists.

In [ ]:
# Branch based on header presence
$var(has_history) = "no";

if ($var(has_history) == "no") {
    xlog("No History-Info header — adding one");
    append_hf("History-Info: <sip:$ru>;index=1");
} else {
    xlog("History-Info already present");
}

## 4. subst — Regex Substitution

`subst("/pattern/replacement/flags")` performs regex substitution across the entire SIP message.

Practical use: domain changes, URI rewriting, etc.

```kamailio
# Change example.com → new-domain.com
subst("/example.com/new-domain.com/g");
```

In [ ]:
# Domain substitution simulation
$var(original) = "sip:1001@old.example.com";
xlog("Original URI: $var(original)");

# Change to new domain
$ru = "sip:1001@new.example.com";
xlog("Rewritten URI: $ru");

subst("/old.example.com/new.example.com/");
xlog("Domain rewrite applied");

## 5. Regex Condition Matching (=~)

The `=~` operator performs regex matching. Useful for checking URI patterns, phone number formats, etc.

In [ ]:
%%sip INVITE
From: <sip:0212345678@example.com>;tag=r1
To: <sip:0211112222@example.com>

In [ ]:
# Check caller number pattern
$var(caller) = $(fu{uri.user});
xlog("Caller number: $var(caller)");

# Area code check (02 prefix = Seoul)
if ($var(caller) =~ "^02") {
    xlog("Seoul area code detected");
} else {
    xlog("Other area code");
}

In [ ]:
# Callee routing branch
$var(callee) = $(ru{uri.user});
xlog("Callee number: $var(callee)");

# International call (00 prefix)
if ($var(callee) =~ "^00") {
    xlog("International call — routing to gateway");
} else {
    # Seoul area code
    if ($var(callee) =~ "^02") {
        xlog("Seoul local call");
    } else {
        xlog("Other domestic call");
    }
}

## 6. Phone Number Normalization

One of the most common tasks in production. Converting various phone number formats
to a standard format.

```
010-1234-5678  →  01012345678
+82-10-1234-5678  →  01012345678
821012345678  →  01012345678
```

In [ ]:
# Number normalization simulation
$var(raw_number) = "+82-10-1234-5678";
xlog("Raw input: $var(raw_number)");

# In production, use subst; here we simulate the result
$var(normalized) = "01012345678";
xlog("Normalized: $var(normalized)");

# Update R-URI
$ru = "sip:$var(normalized)@example.com";
xlog("Updated R-URI: $ru");

## 7. Practical Debugging Techniques

After header manipulation, use `xlog` and variable display to verify changes.

In [ ]:
# Debugging: compare before/after
%%sip INVITE
From: <sip:1001@example.com>;tag=dbg1
To: <sip:1002@example.com>

In [ ]:
# Before
xlog("=== BEFORE ===");
xlog("From: $fu");
xlog("To: $tu");
xlog("R-URI: $ru");

# Header manipulation
append_hf("X-Debug: testing");
$ru = "sip:1002@10.0.0.1:5060";

# After
xlog("=== AFTER ===");
xlog("R-URI: $ru");
xlog("Headers modified — ready to relay");

In [ ]:
# Check variable state
%%vars

---

### Summary

| Concept | Description |
|---------|-------------|
| append_hf("hdr: val") | Add SIP header |
| remove_hf("hdr") | Remove SIP header |
| is_present_hf("hdr") | Check header presence |
| subst("/old/new/") | Regex substitution |
| =~ | Regex match |
| xlog | Debug output |
| %%vars | Display all variable state |

You have completed the full curriculum!